In [0]:
%run ../bronze/bronze_dq_checks

In [0]:
from pyspark.sql import DataFrame, Window, functions as F


In [0]:
date_formats = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy", "yyyy/MM/dd", "MMM d, yyyy"]

PARSED_DATE = "coalesce("+", ".join(
    f"try_to_timestamp(trim(order_date), '{f}')" for f in date_formats
)+ ")"


NULL_TOKENS = ("", "NULL", "N/A", "NA", "NONE", "NAN")
EMAIL_REGEX = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+[.][A-Za-z]{2,}$'

COUNTRY_MAP = {
    "in": "India", "india": "India",
    "us": "United States", "usa": "United States", "united states": "United States", "united states of america": "United States",
    "uk": "UNited Kingdom", "united kingdom": "United Kingdom",
    "uae": "United Arab Emirates"
}


PAYMENT_MAP = {
    "credit card": "Credit Card",
    "debit card": "Debit Card",
    "upi": "UPI",
    "cod": "Cash on Delivery", "cash on delivery": "Cash on Delivery",
    "net banking": "Net Banking",
    "paypal": "PayPal"
}


VALID_STATUSES = ("Delivered", "Shipped", "Processing", "Cancelled", "Returned")

NON_REVENUE_STATUSES = {"Cancelled", "Returned"}


In [0]:
def blank_to_null(c):
    col = F.trim(F.col(c) if isinstance(c, str) else c)
    return F.when(F.upper(col).isin(*NULL_TOKENS), F.lit(None)).otherwise(col)


def title_case(col):
    return F.initcap(F.regexp_replace(col, r"\s+", " "))

def map_values(col, mapping: dict, default):
    expr = None
    for key, value in mapping.items():
        expr = F.when(col == key, F.lit(value)) if expr is None else expr.when(col== key, F.lit(value))

    return expr.otherwise(default)

def normalize_phone(raw):
    digits = F.regexp_replace(blank_to_null(raw), r"[^0-9]", "")
    return (
        F.when(digits.rlike("^0*$"), F.lit(None))
        .when(F.length(digits) < 7, F.lit(None))
        .when((F.length(digits) == 12) & digits.startswith("91"), F.substring(digits, 3, 10))
        .otherwise(digits)
    )



In [0]:
def clean_columns(df: DataFrame):
    name = title_case(blank_to_null("customer_name"))
    email_raw = F.lower(blank_to_null("customer_email"))
    email = F.when(email_raw.rlike(EMAIL_REGEX), email_raw)
    phone =  normalize_phone("phone_number")
    city = title_case(blank_to_null("city"))

    country_raw = blank_to_null("country")
    country = map_values(F.lower(country_raw), COUNTRY_MAP, title_case(country_raw))

    payment_raw = blank_to_null("payment_method")
    payment_key = F.lower(F.regexp_replace(payment_raw, r"[_\s]+", " "))
    payment = map_values(payment_key, PAYMENT_MAP, F.initcap(payment_key))

    status_raw = F.initcap(blank_to_null("order_status"))
    status = F.when(status_raw.isin(*VALID_STATUSES), status_raw).otherwise(F.lit("unknown"))

    discount_raw = F.expr("try_cast(discount_percent AS DECIMAL(5, 2))")
    discount = F.when(discount_raw.between(0, 100), discount_raw).otherwise(F.lit(0).cast("decimal(5, 2)"))
    rating_raw = F.expr("try_cast(rating AS INT)")
    rating= F.when(rating_raw.between(1, 5), rating_raw)

    flags = F.filter(
        F.array(
            F.when(name.isNull(), F.lit("missing_name")),
            F.when(email_raw.isNull(), F.lit("missing_email"))
            .when(~email_raw.rlike(EMAIL_REGEX), F.lit("invalid_email")),
            F.when(phone.isNull(), F.lit("missin_phone")),
            F.when(city.isNull(), F.lit("missing_city")),
            F.when(discount_raw.isNull(), F.lit("missing_discount"))
            .when(~discount_raw.between(0, 100), F.lit("invalid_discount")),
            F.when(rating_raw.isNotNull() & ~rating_raw.between(1, 5), F.lit("invalid_rating")),
            F.when(status == "Unkown", F.lit("unknown_status"))
        ),
        lambda x: x.isNotNull()
    )
    
    return df.select(
        F.col("record_id").alias("bronze_record_id"),
        F.upper(blank_to_null("order_id")).alias("order_id"),
        F.col("order_date").alias("raw_order_date"),
        F.expr(PARSED_DATE).cast("date").alias("order_date"),
        name.alias("customer_name"),
        email.alias("customer_email"),
        phone.alias("phone_number"),
        city.alias("city"),
        country.alias("country"),
        blank_to_null("product_name").alias("product_name"),
        title_case(blank_to_null("category")).alias("category"),
        F.expr("try_cast(quantity AS INT)").alias("quantity"),
        F.expr("try_cast(unit_price AS DECIMAL(10, 2))").alias("unit_price"),
        discount.alias("discount_percent"),
        status.alias("order_status"),
        rating.alias("rating"),
        flags.alias("dq_flags"),
        F.col("_ingestTimeStamp"),
        F.col("sourceFileLocation")
    )

In [0]:
def deduplicate_latest(df: DataFrame, key: str = "order_id") -> DataFrame:
    safe_key = F.coalesce(F.col(key), F.concat(F.lit("__null_"),
F.col("bronze_record_id").cast("string")))
    
    w = Window.partitionBy(safe_key).orderBy(F.col("_ingestTimeStamp").desc(),
F.col("bronze_record_id").desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")
    

In [0]:
def split_valid_rejected(df: DataFrame):
    reasons = F.filter(
        F.array(
            F.when(F.col("order_id").isNull(), F.lit("missing_order_id")),
            F.when(F.col("order_date").isNull(), F.lit("invalid_order_date")),
            F.when(F.col("order_date") > F.current_date(), F.lit("future_order_date")),
            F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), F.lit("invalid_quantity")),
            F.when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), F.lit("invalid_unit_price")),
        ),
        lambda x: x.isNotNull(),
    )
    tagged = df.withColumn("_reasons", reasons)
 
    valid = tagged.where(F.size("_reasons") == 0).drop("_reasons", "raw_order_date")
    rejected = (
        tagged.where(F.size("_reasons") > 0)
              .withColumn("reject_reason", F.concat_ws("; ", "_reasons"))
              .withColumn("_rejected_at", F.current_timestamp())
              .drop("_reasons")
    )
    return valid, rejected

In [0]:
def add_derived_columns(df: DataFrame) -> DataFrame:
    gross = (F.col("quantity") * F.col("unit_price")).cast("decimal(12,2)")
    return (
        df.withColumn("gross_amount", gross)
          .withColumn("discount_amount",
                      F.round(F.col("gross_amount") * F.col("discount_percent") / 100, 2).cast("decimal(12,2)"))
          .withColumn("net_amount", (F.col("gross_amount") - F.col("discount_amount")).cast("decimal(12,2)"))
          .withColumn("order_year", F.year("order_date"))
          .withColumn("order_month", F.month("order_date"))
          .withColumn("price_band",
                      F.when(F.col("unit_price") < 1000, "Low")
                       .when(F.col("unit_price") < 5000, "Medium")
                       .otherwise("High"))
          .withColumn("is_revenue_order", ~F.col("order_status").isin(*NON_REVENUE_STATUSES))
    )
 

In [0]:
 
def transform(bronze_df: DataFrame):
    cleaned = clean_columns(bronze_df)
    latest = deduplicate_latest(cleaned)
    valid, rejected = split_valid_rejected(latest)
    valid = add_derived_columns(valid).withColumn("_silver_processed_at", F.current_timestamp())
    return valid, rejected
 
 
def filter_new_rows(df: DataFrame, watermark) -> DataFrame:
    """Incremental load: only bronze rows ingested after the last processed timestamp."""
    if watermark is None:
        return df
    return df.where(F.col("_ingestTimeStamp") > F.lit(watermark))
 